# 과제 7 — CNN 이미지 분류

## 가상 이미지 데이터셋 생성

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# 가상 이미지 데이터셋 — PyTorch 형식 (N, C, H, W)
num_classes = 10
X = np.random.random((2000, 3, 32, 32)).astype(np.float32)
y = np.random.randint(num_classes, size=2000)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'학습: {X_train_np.shape}, 테스트: {X_test_np.shape}')

## CNN 모델 구성

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),   nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN(num_classes)
print(model)

## 학습 및 평가

In [ ]:
X_tr = torch.from_numpy(X_train_np)
y_tr = torch.from_numpy(y_train_np)
X_te = torch.from_numpy(X_test_np)
y_te = torch.from_numpy(y_test_np)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=64)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

for epoch in range(10):
    model.train()
    train_loss, train_correct = 0, 0
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        out  = model(X_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * len(y_b)
        train_correct += (out.argmax(1) == y_b).sum().item()

    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for X_b, y_b in test_loader:
            out         = model(X_b)
            val_loss    += criterion(out, y_b).item() * len(y_b)
            val_correct += (out.argmax(1) == y_b).sum().item()

    n_tr, n_te = len(y_tr), len(y_te)
    history['loss'].append(train_loss / n_tr)
    history['accuracy'].append(train_correct / n_tr)
    history['val_loss'].append(val_loss / n_te)
    history['val_accuracy'].append(val_correct / n_te)

    print(f'Epoch {epoch+1:2d}/10  '
          f'loss: {history["loss"][-1]:.4f}  acc: {history["accuracy"][-1]:.4f}  '
          f'val_loss: {history["val_loss"][-1]:.4f}  val_acc: {history["val_accuracy"][-1]:.4f}')

In [ ]:
print(f'테스트 정확도: {history["val_accuracy"][-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['accuracy'], label='train')
axes[1].plot(history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()